# 02 · Đặc trưng, dữ liệu thiếu và kiểm tra chất lượng (phụ trách: Khoẻ · Chiến)

**Đề:** *"distributed feature-engineering"* + *"missing-data sensitivity"*.
Mã nguồn: `data_spark/features.py`, `data_spark/features_discharge.py`, `data_spark/quality.py`, `contracts/gold_schema_*.json`.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

## 1. Hợp đồng Gold: 114 đặc trưng P1, 146 đặc trưng P2/P3

In [ ]:
for t in ("p1", "p2"):
    s = json.loads((ROOT / f"contracts/gold_schema_{t}.json").read_text(encoding="utf-8"))
    groups = {}
    for g, spec in s["feature_columns"].items():
        groups[g] = len(spec) if isinstance(spec, list) else len(spec["items"]) * len(spec["suffixes"]) + len(spec.get("extra", []))
    print(t, groups, "tổng", sum(groups.values()), "| cột cấm:", s["forbidden_columns"])

## 2. Tỷ lệ thiếu thật (P1) — thiếu không ngẫu nhiên

In [ ]:
pd.Series({k: v for k, v in VERIFIED['missing_rate_p1'].items() if not isinstance(v, list)}).sort_values(ascending=False).to_frame('tỷ lệ thiếu')

## 3. Lỗi đọc dữ liệu đã sửa ở Dataset v1.1
`"<0.01"` là kết quả **bình thường dưới ngưỡng phát hiện**, không phải thiếu.

In [ ]:
pd.DataFrame(VERIFIED['data_issues_fixed_in_v1_1'])

In [ ]:
# Hàm đọc giá trị bị chặn (Spark). Chạy được nếu máy có Java 17.
import re
from data_spark.features import CENSORED_RE
for v in ["<0.01", "< 0.03", ">60", "0.25", "NEG"]:
    m = re.match(CENSORED_RE, v)
    print(f"{v!r:10} → {float(m.group(2)) if m else 'giữ nguyên/thiếu'}")

## 4. Chống rò rỉ: những gì bị cấm và vì sao

In [ ]:
s1 = json.loads((ROOT / "contracts/gold_schema_p1.json").read_text(encoding="utf-8"))
print("Cấm ở P1:", s1["forbidden_columns"])
print("P1 dùng xét nghiệm trong [admittime − 6h, admittime + 24h], chỉ khi storetime ≤ mốc dự đoán.")

## Câu hỏi tự kiểm tra
1. `charttime` và `storetime` khác nhau thế nào? Vì sao lọc theo `storetime`?
2. Vì sao ICD của lượt hiện tại bị cấm ở P1 nhưng được dùng ở P2/P3?
3. Cờ `_missing` có ý nghĩa gì với mô hình cây? Có rủi ro gì (gợi ý: phân tích C5)?
4. Tỷ lệ thiếu troponin trước và sau v1.1 là bao nhiêu? (đọc `results/data/features_result_full.md` sau khi chạy lại)

**Chạy thật:** `make etl TASK=p1 VERSION=v1_1` rồi `make quality TASK=p1 VERSION=v1_1`.